In [1]:
!pip install -q langchain langgraph langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.0 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [2]:
!git clone https://github.com/JuanBurgos-BTF/neuro-agente-triaje.git
%cd neuro-agente-triaje
!ls models

Cloning into 'neuro-agente-triaje'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 14 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 28.37 MiB | 14.00 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/neuro-agente-triaje
modelo_base.keras  modelo_mejorado.keras


In [3]:
import numpy as np
import tensorflow as tf

modelo = tf.keras.models.load_model("models/modelo_base.keras")

resena = np.array([
    "I watched this movie last weekend with my family. The acting was excellent, "
    "the story kept me engaged from start to finish, and the ending was beautiful. "
    "I would definitely recommend it to anyone who loves a good drama."
], dtype=object)

print("P(positivo):", float(modelo.predict(resena, verbose=0)[0][0]))

P(positivo): 0.7733637094497681


In [4]:
import os
os.makedirs("src", exist_ok=True)

open("src/__init__.py", "w").write("")

open("src/modelo.py", "w").write('''
import numpy as np
import tensorflow as tf

_MODELO = None

def cargar_modelo(ruta="models/modelo_base.keras"):
    global _MODELO
    if _MODELO is None:
        _MODELO = tf.keras.models.load_model(ruta)
    return _MODELO

def predecir_sentimiento(texto: str) -> dict:
    m = cargar_modelo()
    p = float(m.predict(np.array([texto], dtype=object), verbose=0)[0][0])
    etiqueta = "POSITIVO" if p >= 0.5 else "NEGATIVO"
    confianza = p if p >= 0.5 else 1 - p
    return {"etiqueta": etiqueta, "p_positivo": round(p, 4), "confianza": round(confianza, 4)}
''')

open("src/herramientas.py", "w").write('''
from langchain_core.tools import tool
from src.modelo import predecir_sentimiento

@tool
def clasificar_sentimiento(texto: str) -> str:
    """Clasifica con la red neuronal local (Embedding+Dense, entrenada en IMDB) el
    sentimiento de UNA reseña en INGLES. Usala solo cuando el usuario entregue el
    texto de una reseña o ticket para analizar. Devuelve etiqueta (POSITIVO/NEGATIVO)
    y la probabilidad. No sirve para saludos ni preguntas sobre politicas."""
    r = predecir_sentimiento(texto)
    return f"{r['etiqueta']} | P(positivo)={r['p_positivo']} | confianza={r['confianza']}"

@tool
def consultar_politica_escalamiento(etiqueta: str, confianza: float) -> str:
    """Aplica las reglas de escalamiento del equipo de soporte. Recibe la etiqueta
    (POSITIVO o NEGATIVO) y la confianza (0 a 1) de la clasificacion. Usala para
    decidir que accion tomar o cuando el usuario pregunte por las politicas de
    escalamiento. No clasifica texto."""
    etiqueta = etiqueta.upper().strip()
    if confianza < 0.60:
        return "ESCALAR A REVISION HUMANA: confianza menor a 0.60, el modelo no es fiable en este caso."
    if etiqueta == "NEGATIVO" and confianza >= 0.90:
        return "PRIORIDAD ALTA: responder en menos de 2 horas y notificar al supervisor."
    if etiqueta == "NEGATIVO":
        return "PRIORIDAD MEDIA: responder en 24 horas con el protocolo estandar de disculpa."
    return "PRIORIDAD BAJA: agradecer al cliente y archivar."
''')

open("src/agente.py", "w").write('''
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
from src.herramientas import clasificar_sentimiento, consultar_politica_escalamiento

SYSTEM_PROMPT = (
    "Eres un agente de triaje de resenas y tickets. Responde en espanol y de forma breve. "
    "Si el usuario solo saluda o hace una pregunta general, responde sin usar herramientas. "
    "Si entrega una resena para analizar, usa clasificar_sentimiento. "
    "Para decidir la accion, usa consultar_politica_escalamiento con la etiqueta y confianza obtenidas."
)

def crear_agente(modelo_llm=None):
    llm = ChatGoogleGenerativeAI(
        model=modelo_llm or os.getenv("GEMINI_MODEL", "gemini-2.5-flash"),
        temperature=0,
        max_output_tokens=512,
    )
    return create_agent(
        llm,
        tools=[clasificar_sentimiento, consultar_politica_escalamiento],
        system_prompt=SYSTEM_PROMPT,
        checkpointer=MemorySaver(),
    )
''')

print(os.listdir("src"))

['herramientas.py', '__init__.py', 'modelo.py', 'agente.py']


In [5]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

from src.agente import crear_agente
agente = crear_agente()

def preguntar(texto, thread="t1"):
    r = agente.invoke({"messages": [("user", texto)]},
                      config={"configurable": {"thread_id": thread}})
    return r["messages"]

msgs = preguntar("Hola, ¿qué puedes hacer?")
print(msgs[-1].content)

GoogleAuthenticationError: Error calling model 'gemini-2.5-flash' (UNAUTHENTICATED): 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_TYPE_UNSUPPORTED', 'metadata': {'service': 'generativelanguage.googleapis.com', 'method': 'google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent'}}]}}

In [6]:
from google.colab import userdata
from google import genai

k = userdata.get("GOOGLE_API_KEY").strip()
print(k[:3], len(k))

client = genai.Client(api_key=k)
r = client.models.generate_content(model="gemini-2.5-flash", contents="Di hola en una palabra")
print(r.text)

AQ. 105


ClientError: 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_TYPE_UNSUPPORTED', 'metadata': {'method': 'google.ai.generativelanguage.v1.GenerativeService.GenerateContent', 'service': 'generativelanguage.googleapis.com'}}]}}

In [7]:
from google.colab import userdata
from google import genai

k = userdata.get("GOOGLE_API_KEY").strip()
client = genai.Client(vertexai=True, api_key=k)
r = client.models.generate_content(model="gemini-2.5-flash", contents="Di hola en una palabra")
print(r.text)

ClientError: 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_TYPE_UNSUPPORTED', 'metadata': {'method': 'google.cloud.aiplatform.v1beta1.PredictionService.GenerateContent', 'service': 'aiplatform.googleapis.com'}}]}}

In [8]:
from google.colab import userdata
from google import genai

# Obtener la nueva clave actualizada
k = userdata.get("GOOGLE_API_KEY").strip()
print(f"Nueva longitud de la clave: {len(k)} caracteres")

try:
    # Inicializar el cliente estándar
    client = genai.Client(api_key=k)
    r = client.models.generate_content(model="gemini-2.5-flash", contents="Di 'Listo' en una palabra")
    print("Resultado de la prueba:", r.text.strip())
    print("¡Verificación exitosa! Tu API Key está funcionando correctamente.")
except Exception as e:
    print("Aún hay un problema con la clave:", e)

Nueva longitud de la clave: 53 caracteres
Aún hay un problema con la clave: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}


In [11]:
from google.colab import userdata
from google import genai

k = userdata.get("GOOGLE_API_KEY").strip()
client = genai.Client(api_key=k)

try:
    # Ahora que la clave está corregida, probamos el modelo disponible y soportado en tu cuenta
    r = client.models.generate_content(model="gemini-2.5-flash", contents="Di 'Listo' en una palabra")
    print("Resultado de la prueba:", r.text.strip())
    print("¡Verificación exitosa! Tu API Key y el modelo gemini-2.5-flash están listos para usarse.")
except Exception as e:
    print("Error al intentar conectar:", e)

--- 1. Probando con gemini-2.0-flash (Recomendado actual) ---
Error con gemini-2.0-flash: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

--- 2. Listando modelos disponibles para tu cuenta ---
ID: models/gemini-2.5-flash | Soportado: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
ID: models/gemini-2.5-pro | Soportado: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
ID: models/gemma-4-26b-a4b-it | Soportado: ['generateContent', 'countTokens']
ID: models/gemma-4-31b-it | Soportado: ['generateContent', 'countTokens']
ID: models/gemini-2.5-flash-lite | Soportado: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
ID: 

In [13]:
from google.colab import userdata
from google import genai

k = userdata.get("GOOGLE_API_KEY").strip()
client = genai.Client(api_key=k)

# Usamos gemini-3.1-flash-lite, que está activo y listado en tu cuenta de forma gratuita
try:
    r = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents="Di hola en una palabra"
    )
    print("Respuesta del modelo:", r.text.strip())
    print("¡Conexión y generación exitosas!")
except Exception as e:
    print("Error al conectar:", e)

Respuesta del modelo: Hola.
¡Conexión y generación exitosas!


In [14]:
open("src/agente.py", "w").write('''
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
from src.herramientas import clasificar_sentimiento, consultar_politica_escalamiento

SYSTEM_PROMPT = (
    "Eres un agente de triaje de resenas y tickets. Responde en espanol y de forma breve. "
    "Si el usuario solo saluda o hace una pregunta general, responde sin usar herramientas. "
    "Si entrega una resena para analizar, usa clasificar_sentimiento. "
    "Para decidir la accion, usa consultar_politica_escalamiento con la etiqueta y confianza obtenidas."
)

def crear_agente(modelo_llm=None):
    llm = ChatGoogleGenerativeAI(
        model=modelo_llm or os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite"),
        google_api_key=os.environ["GOOGLE_API_KEY"],
        temperature=0,
        max_output_tokens=512,
    )
    return create_agent(
        llm,
        tools=[clasificar_sentimiento, consultar_politica_escalamiento],
        system_prompt=SYSTEM_PROMPT,
        checkpointer=MemorySaver(),
    )
''')

1085

In [15]:
import os, sys, importlib
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY").strip()

if "src.agente" in sys.modules:
    importlib.reload(sys.modules["src.agente"])
from src.agente import crear_agente
agente = crear_agente()

def preguntar(texto, thread="t1"):
    r = agente.invoke({"messages": [("user", texto)]},
                      config={"configurable": {"thread_id": thread}})
    return r["messages"]

msgs = preguntar("Hola, ¿qué puedes hacer?")
print(msgs[-1].content)

[{'type': 'text', 'text': 'Hola. Soy un agente de triaje diseñado para analizar reseñas y tickets de clientes. Puedo clasificar el sentimiento de tus textos (positivo o negativo) y determinar, basándome en nuestras políticas, si el caso debe ser escalado a un equipo de soporte humano.\n\n¿Tienes alguna reseña o ticket que necesites que analice?', 'extras': {'signature': 'EqEBCp4BAWkUfRNuhDM9X78g2ZubOSN3khKUIGuLrCgEzOmCtJjsZefXIQ77luL67H2+nubmvkrjnWMDzPw5ra/f2dHkc/GbrpZxPuAqlgXYsi0epPsKFsJsUbbc4JzVTJhs8jk/dzUgX8tmP0jEaUUw8euv59KD/FCfr+hg/kk8HgvMuTITIEm2enPWbrgRV8FOdLqvANPqHBMuOYAz0veG3X8='}}]


In [17]:
import json

def texto(msg):
    c = msg.content
    if isinstance(c, list):
        return "".join(p.get("text", "") for p in c if isinstance(p, dict))
    return c

def herramientas_usadas(msgs):
    usadas = []
    for m in msgs:
        for tc in (getattr(m, "tool_calls", None) or []):
            usadas.append(tc["name"])
    return usadas

RESENA_POS = ("I watched this movie last weekend with my family. The acting was excellent, "
              "the story kept me engaged from start to finish, and the ending was beautiful. "
              "I would definitely recommend it to anyone who loves a good drama.")
RESENA_NEG = ("This was the worst movie I have ever seen. The plot made no sense, the acting was "
              "terrible, and I wasted two hours of my life. I regret buying the ticket and I "
              "would never recommend this boring disaster to anybody.")

casos = [
    {"id": 1, "desc": "Saludo (sin herramientas)", "thread": "eval-1",
     "entrada": "Hola, ¿qué puedes hacer?", "esperadas": []},
    {"id": 2, "desc": "Solo politica (tool 2)", "thread": "eval-2",
     "entrada": "¿Qué pasa si una reseña es NEGATIVA con confianza 0.95?",
     "esperadas": ["consultar_politica_escalamiento"]},
    {"id": 3, "desc": "Completo: red + politica", "thread": "eval-3",
     "entrada": "Analiza esta reseña y dime qué acción tomar: " + RESENA_NEG,
     "esperadas": ["clasificar_sentimiento", "consultar_politica_escalamiento"]},
    {"id": 4, "desc": "Solo red (tool 1)", "thread": "eval-4",
     "entrada": "Clasifica el sentimiento de esta reseña: " + RESENA_POS,
     "esperadas": ["clasificar_sentimiento"]},
]

filas = []
for c in casos:
    msgs = preguntar(c["entrada"], thread=c["thread"])
    usadas = herramientas_usadas(msgs)
    ok = sorted(set(usadas)) == sorted(set(c["esperadas"]))
    filas.append({"caso": c["id"], "descripcion": c["desc"],
                  "esperadas": c["esperadas"], "invocadas": usadas,
                  "trayectoria_ok": ok})
    print(f"\n=== CASO {c['id']}: {c['desc']} ===")
    print("Esperadas:", c["esperadas"], "| Invocadas:", usadas, "| ¿OK?:", ok)


=== CASO 1: Saludo (sin herramientas) ===
Esperadas: [] | Invocadas: [] | ¿OK?: True

=== CASO 2: Solo politica (tool 2) ===
Esperadas: ['consultar_politica_escalamiento'] | Invocadas: ['consultar_politica_escalamiento'] | ¿OK?: True

=== CASO 3: Completo: red + politica ===
Esperadas: ['clasificar_sentimiento', 'consultar_politica_escalamiento'] | Invocadas: ['clasificar_sentimiento', 'consultar_politica_escalamiento'] | ¿OK?: True

=== CASO 4: Solo red (tool 1) ===
Esperadas: ['clasificar_sentimiento'] | Invocadas: ['clasificar_sentimiento', 'consultar_politica_escalamiento'] | ¿OK?: False


In [18]:
p = open("src/agente.py").read()
p = p.replace(
 '"Para decidir la accion, usa consultar_politica_escalamiento con la etiqueta y confianza obtenidas."',
 '"Usa consultar_politica_escalamiento SOLO si el usuario pide expresamente una accion, prioridad o escalamiento. Si solo pide clasificar, responde unicamente con la clasificacion."')
open("src/agente.py", "w").write(p)

import sys, importlib
importlib.reload(sys.modules["src.agente"])
from src.agente import crear_agente
agente = crear_agente()
print("Cambio aplicado:", "SOLO si el usuario pide" in open("src/agente.py").read())

Cambio aplicado: True


In [19]:
preguntar("Mi nombre es Ana.", thread="mem-A")
print("Mismo thread:", texto(preguntar("¿Cómo me llamo?", thread="mem-A")[-1]))
print("Otro thread:", texto(preguntar("¿Cómo me llamo?", thread="mem-B")[-1]))

Mismo thread: Te llamas Ana.
Otro thread: No tengo acceso a tu información personal, por lo que no sé cómo te llamas. ¿En qué más puedo ayudarte?


In [20]:
RESENA_POS = ("I watched this movie last weekend with my family. The acting was excellent, "
              "the story kept me engaged from start to finish, and the ending was beautiful. "
              "I would definitely recommend it to anyone who loves a good drama.")
RESENA_NEG = ("This was the worst movie I have ever seen. The plot made no sense, the acting was "
              "terrible, and I wasted two hours of my life. I regret buying the ticket and I "
              "would never recommend this boring disaster to anybody.")

def texto(msg):
    c = msg.content
    if isinstance(c, list):
        return "".join(p.get("text", "") for p in c if isinstance(p, dict))
    return c

def herramientas_usadas(msgs):
    return [tc["name"] for m in msgs for tc in (getattr(m, "tool_calls", None) or [])]

casos = [
    {"id": 1, "desc": "Saludo (sin herramientas)", "thread": "eval-1",
     "entrada": "Hola, ¿qué puedes hacer?", "esperadas": []},
    {"id": 2, "desc": "Solo politica (tool 2)", "thread": "eval-2",
     "entrada": "¿Qué pasa si una reseña es NEGATIVA con confianza 0.95?",
     "esperadas": ["consultar_politica_escalamiento"]},
    {"id": 3, "desc": "Completo: red + politica", "thread": "eval-3",
     "entrada": "Analiza esta reseña y dime qué acción tomar: " + RESENA_NEG,
     "esperadas": ["clasificar_sentimiento", "consultar_politica_escalamiento"]},
    {"id": 4, "desc": "Solo red (tool 1)", "thread": "eval-4",
     "entrada": "Clasifica el sentimiento de esta reseña: " + RESENA_POS,
     "esperadas": ["clasificar_sentimiento"]},
]

for c in casos:
    msgs = preguntar(c["entrada"], thread=c["thread"])
    usadas = herramientas_usadas(msgs)
    ok = sorted(set(usadas)) == sorted(set(c["esperadas"]))
    print(f"\n=== CASO {c['id']}: {c['desc']} ===")
    print("Esperadas:", c["esperadas"], "| Invocadas:", usadas, "| OK:", ok)
    print("Respuesta:", texto(msgs[-1])[:300])


=== CASO 1: Saludo (sin herramientas) ===
Esperadas: [] | Invocadas: [] | OK: True
Respuesta: Hola. Puedo ayudarte a clasificar el sentimiento de reseñas o tickets de soporte en inglés (positivo o negativo) y orientarte sobre las políticas de escalamiento de nuestro equipo si lo necesitas. ¿En qué puedo ayudarte hoy?

=== CASO 2: Solo politica (tool 2) ===
Esperadas: ['consultar_politica_escalamiento'] | Invocadas: ['consultar_politica_escalamiento'] | OK: True
Respuesta: Si una reseña es clasificada como NEGATIVA con una confianza de 0.95, la política de escalamiento indica que se debe tratar con **prioridad alta**: se debe responder en menos de 2 horas y notificar al supervisor.

=== CASO 3: Completo: red + politica ===
Esperadas: ['clasificar_sentimiento', 'consultar_politica_escalamiento'] | Invocadas: ['clasificar_sentimiento', 'consultar_politica_escalamiento'] | OK: True
Respuesta: La reseña es NEGATIVA (confianza: 99.31%). Debido a esto, la política de escalamiento indica: PRI